In [1]:
!pip install -q -U torchao pylcs faiss-cpu rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 125.5 MB/s eta 0:00:00


In [2]:
# =============================================================================
# BOOTSTRAP: restore full pipeline state from Drive caches (~5-10 min, CPU ok)
# Run this first after ANY runtime restart.
# =============================================================================
import os, re as _re, json, pickle, gc
import numpy as np, pandas as pd, faiss
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from rouge_score import rouge_scorer
from rouge_score import tokenize as rs_tokenize
try:
    import pylcs; HAVE_PYLCS = True
except ImportError:
    os.system('pip install -q pylcs')
    try: import pylcs; HAVE_PYLCS = True
    except Exception: HAVE_PYLCS = False

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/multilingual-health-qa')
DATA_DIR, OUTPUT_DIR = BASE/'data', BASE/'outputs'
CACHE = OUTPUT_DIR/'mbr_cache'

# ---- data ----
train_df = pd.read_csv(DATA_DIR/'Train.csv')
val_df   = pd.read_csv(DATA_DIR/'Val.csv')
test_df  = pd.read_csv(DATA_DIR/'Test.csv')
sample_sub = pd.read_csv(DATA_DIR/'SampleSubmission.csv'); SUB_COLS = list(sample_sub.columns)
FT_MODEL_DIR = OUTPUT_DIR / 'qwen-ft-health'
combined = pd.concat([train_df, val_df], ignore_index=True).dropna(subset=['input','output']).reset_index(drop=True)
questions_raw = combined['input'].astype(str).tolist()
answers_raw   = combined['output'].astype(str).tolist()
subsets_raw   = combined['subset'].astype(str).tolist()
corpus_q_stripped = [q.strip() for q in questions_raw]
val_qs  = val_df['input'].fillna('').astype(str).tolist()
test_qs = test_df['input'].fillna('').astype(str).tolist()
test_subs = test_df['subset'].fillna('').astype(str).tolist()
SUBSET_TO_LANG = {'Aka_Gha':'Akan (Ghana)','Amh_Eth':'Amharic (Ethiopia)','Eng_Eth':'English (Ethiopia)',
 'Eng_Gha':'English (Ghana)','Eng_Ken':'English (Kenya)','Eng_Uga':'English (Uganda)',
 'Lug_Uga':'Luganda (Uganda)','Swa_Ken':'Swahili (Kenya)'}
scorer_both = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=False)

# ---- embeddings (all cached) ----
corpus_emb = np.load(CACHE/'emb_corpus.npy'); val_emb = np.load(CACHE/'emb_val.npy'); test_emb = np.load(CACHE/'emb_test.npy')
gem_corpus = np.load(CACHE/'gem_emb_corpus.npy'); gem_val = np.load(CACHE/'gem_emb_val.npy'); gem_test = np.load(CACHE/'gem_emb_test.npy')
ans_emb = np.load(CACHE/'emb_corpus_answers.npy')
bge_corpus = np.load(CACHE/'bge_corpus.npy'); bge_val = np.load(CACHE/'bge_val.npy'); bge_test = np.load(CACHE/'bge_test.npy')

# ---- indices ----
def build_lang_idx(emb):
    out = {}
    for sub in sorted(set(subsets_raw)):
        mask = [i for i,s in enumerate(subsets_raw) if s==sub]
        ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb[mask]); out[sub]=(ix,mask)
    return out
lang_indices = build_lang_idx(corpus_emb); gem_lang_idx = build_lang_idx(gem_corpus)
qa_idx = build_lang_idx(ans_emb); bge_idx = build_lang_idx(bge_corpus)
global_idx = faiss.IndexFlatIP(corpus_emb.shape[1]); global_idx.add(corpus_emb)

# ---- pickled state ----
val_cands_all = pickle.load(open(CACHE/'val_cands.pkl','rb'))
val_prep, val_refscores = pickle.load(open(CACHE/'val_prep.pkl','rb'))
v4c, v4p, v4r = pickle.load(open(CACHE/'val_union4.pkl','rb'))
P = pickle.load(open(CACHE/'uni_rebuild.pkl','rb'))
llm_ans = json.load(open(OUTPUT_DIR/'gemini_mbr_llm_prog.json'))

# ---- core functions (canonical definitions) ----
K_CANDIDATES, K_LEG, CAP = 15, 20, 400
_UNI = _re.compile(r'\w+', _re.UNICODE)
def uni_toks(t): return _UNI.findall(t.lower())
def _lcs_py(a,b):
    if not a or not b: return 0
    dp=[0]*(len(b)+1)
    for ai in a:
        prev=0
        for j,bj in enumerate(b):
            cur=dp[j+1]; dp[j+1]=prev+1 if ai==bj else max(dp[j+1],dp[j]); prev=cur
    return dp[-1]
def lcs_tok(a,b):
    if HAVE_PYLCS:
        v={}
        for t in a:
            if t not in v: v[t]=len(v)
        for t in b:
            if t not in v: v[t]=len(v)
        return pylcs.lcs_sequence_length(''.join(chr(0x100+v[t]) for t in a), ''.join(chr(0x100+v[t]) for t in b))
    return _lcs_py(a,b)
def uni_r1(rt,ht):
    if not rt or not ht: return 0.0
    return 2*sum((Counter(rt)&Counter(ht)).values())/(len(rt)+len(ht))
def uni_rl(rt,ht):
    if not rt or not ht: return 0.0
    return 2*lcs_tok(rt,ht)/(len(rt)+len(ht))
def get_same_lang_candidates(q_text,q_emb,subset,k=K_CANDIDATES,exclude_exact=True):
    qs=q_text.strip()
    if subset in lang_indices:
        idx,mask=lang_indices[subset]
        D,I=idx.search(np.asarray(q_emb,np.float32).reshape(1,-1),min(k+5,len(mask)))
        out=[]
        for d,li in zip(D[0],I[0]):
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            out.append({'answer':answers_raw[ci],'sim':float(d),'idx':ci})
            if len(out)>=k: break
        if out: return out
    return []
def union4(q_text,afri_q,gem_q,bge_q,subset,exclude_exact=True):
    qs=q_text.strip(); rrf={}
    for idx_map,emb in [(lang_indices,afri_q),(gem_lang_idx,gem_q),(qa_idx,afri_q),(bge_idx,bge_q)]:
        if subset not in idx_map: continue
        idx,mask=idx_map[subset]
        D,I=idx.search(np.asarray(emb,np.float32).reshape(1,-1),min(K_LEG+5,len(mask)))
        r=0
        for li in I[0]:
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            rrf[ci]=rrf.get(ci,0.0)+1.0/(60+r); r+=1
            if r>=K_LEG: break
    ranked=sorted(rrf.items(),key=lambda kv:-kv[1])[:35]
    return [{'answer':answers_raw[ci],'sim':float(np.dot(afri_q,corpus_emb[ci])),'idx':ci} for ci,_ in ranked]
def uni_prep(cands,max_tok=80):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    seen,dd,ddw={},[],[]
    for a,wi in zip(answers,w):
        k=a.strip().lower()
        if k in seen: ddw[seen[k]]+=wi
        else: seen[k]=len(dd); dd.append(a); ddw.append(wi)
    ddw=np.array(ddw); ddw/=ddw.sum(); n=len(dd)
    toks=[uni_toks(a)[:max_tok] for a in dd]
    if n==1: return dd,ddw,np.zeros(1),np.zeros(1)
    S1,SL=np.zeros((n,n)),np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):
            S1[i,j]=S1[j,i]=uni_r1(toks[i],toks[j]); SL[i,j]=SL[j,i]=uni_rl(toks[i],toks[j])
    return dd,ddw,S1@ddw,SL@ddw
def mbr_idx(ub,ddw,alpha,margin):
    if len(ddw)==1: return 0
    u=ub+alpha*ddw; b=int(np.argmax(u))
    return b if (b!=0 and u[b]-u[0]>margin) else 0
uni_prior={sub: float(np.median([len(uni_toks(answers_raw[i])) for i,s in enumerate(subsets_raw) if s==sub])) for sub in SUBSET_TO_LANG}
def uni_stitch(cands,lam,sub):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    p=Counter()
    for a,wi in zip(answers,w):
        for t,c in Counter(uni_toks(a)[:CAP]).items(): p[t]+=wi*c
    ref_len=max(lam*uni_prior[sub],1.0); seen,pool=set(),[]
    for a in answers:
        for s in _re.split(r'(?<=[.!?])\s+|\n+',a):
            s=s.strip(); st=uni_toks(s)
            if len(st)<4: continue
            k=' '.join(st)
            if k not in seen: seen.add(k); pool.append((s,Counter(st),len(st)))
    if not pool: return answers[0]
    def ef1(cov,hl):
        mt=sum(min(c,p[t]) for t,c in cov.items())
        if hl==0 or mt==0: return 0.0
        pr_,rc=mt/hl,mt/ref_len; return 2*pr_*rc/(pr_+rc)
    chosen,cov,hl,cur=[],Counter(),0,0.0
    while pool:
        bi,bg=-1,0.0
        for i,(s,sc_,sl) in enumerate(pool):
            nc=cov.copy(); nc.update(sc_)
            g=ef1(nc,hl+sl)-cur
            if g>bg: bg,bi=g,i
        if bi<0: break
        s,sc_,sl=pool.pop(bi); chosen.append(s); cov.update(sc_); hl+=sl; cur+=bg
    return ' '.join(chosen) if chosen else answers[0]

# ---- tuned decisions from the unicode rebuild (hardcoded = reproducible) ----
choice = {'Aka_Gha':('2leg',0.10,0.010),'Amh_Eth':('2leg',0.05,99.0),'Eng_Eth':('2leg',0.05,99.0),
          'Eng_Gha':('4leg',0.20,0.050),'Eng_Ken':('2leg',0.05,99.0),'Eng_Uga':('2leg',0.05,99.0),
          'Lug_Uga':('2leg',0.05,99.0),'Swa_Ken':('2leg',0.05,99.0)}
uni_stitch_gate = {'Aka_Gha':{'use':True,'lam':0.70,'pool':'2leg'},
                   'Eng_Gha':{'use':True,'lam':0.85,'pool':'4leg'},
                   'Amh_Eth':{'use':True,'lam':0.70,'pool':'2leg'}}
print("STATE RESTORED:", len(combined), "corpus |", len(llm_ans), "LLM answers | all caches loaded")

Mounted at /content/drive
STATE RESTORED: 36501 corpus | 2618 LLM answers | all caches loaded


In [ ]:
# =============================================================================
# CROSS-ENCODER RERANKER v2 — done right: classification, unicode labels, guarded
# Train: XLM-R-base on (query, candidate-question) pairs. Deploy: rerank top-15,
# per-language gate + margin override. L4/T4, ~2-3 hrs total.
# =============================================================================
!pip install -q -U transformers accelerate
import torch, gc, random, numpy as np
from tqdm import tqdm
random.seed(42); np.random.seed(42)

# ---- 1) MINE PAIRS (unicode labels, train rows only, all languages) ----
train_q_set = set(q.strip() for q in train_df['input'].dropna().astype(str))
is_train_row = np.array([q in train_q_set for q in corpus_q_stripped])
pairs = []   # (query_text, cand_text, label)
PER_LANG = 2500
for sub in SUBSET_TO_LANG:
    idx_t = [i for i in range(len(combined)) if subsets_raw[i]==sub and is_train_row[i]]
    random.shuffle(idx_t)
    ix, mask = lang_indices[sub]; mask_arr = np.array(mask)
    tmask = np.array([is_train_row[ci] for ci in mask])
    made = 0
    for qi in idx_t:
        if made >= PER_LANG: break
        gold = uni_toks(answers_raw[qi])[:CAP]
        if len(gold) < 3: continue
        D, I = ix.search(corpus_emb[qi].reshape(1,-1), 20)
        pos, neg = [], []
        for li in I[0]:
            if li < 0 or not tmask[int(li)]: continue
            ci = int(mask_arr[int(li)])
            if ci == qi or corpus_q_stripped[ci] == corpus_q_stripped[qi]: continue
            r1 = uni_r1(gold, uni_toks(answers_raw[ci])[:CAP])
            (pos if r1 >= 0.5 else neg if r1 <= 0.2 else []).append(ci)
        if pos and neg:
            pairs.append((questions_raw[qi], questions_raw[random.choice(pos)], 1))
            for n_ in random.sample(neg, min(2, len(neg))):
                pairs.append((questions_raw[qi], questions_raw[n_], 0))
            made += 1
print(f"Pairs: {len(pairs)}  (pos {sum(1 for p in pairs if p[2]==1)})")

# ---- 2) TRAIN XLM-R-base cross-encoder (binary) ----
from transformers import AutoTokenizer, AutoModelForSequenceClassification
gc.collect(); torch.cuda.empty_cache()
CE = 'xlm-roberta-base'
ctok = AutoTokenizer.from_pretrained(CE)
cmod = AutoModelForSequenceClassification.from_pretrained(CE, num_labels=2).to('cuda:0')
opt = torch.optim.AdamW(cmod.parameters(), lr=2e-5)
random.shuffle(pairs)
B = 16
cmod.train()
for ep in range(1):
    pbar = tqdm(range(0, len(pairs), B), desc=f"CE epoch {ep+1}")
    for s in pbar:
        chunk = pairs[s:s+B]
        enc = ctok([p[0] for p in chunk], [p[1] for p in chunk], padding=True,
                   truncation=True, max_length=160, return_tensors='pt').to('cuda:0')
        labels = torch.tensor([p[2] for p in chunk]).to('cuda:0')
        out = cmod(**enc, labels=labels)
        out.loss.backward(); opt.step(); opt.zero_grad()
        if s % (B*50) == 0: pbar.set_postfix(loss=float(out.loss))
cmod.eval()
cmod.save_pretrained(str(OUTPUT_DIR/'ce-reranker-v2')); ctok.save_pretrained(str(OUTPUT_DIR/'ce-reranker-v2'))

# ---- 3) GUARDED EVAL: rerank top-15, per-language margin gate, split-half ----
@torch.no_grad()
def ce_scores(query, cand_questions):
    enc = ctok([query]*len(cand_questions), cand_questions, padding=True,
               truncation=True, max_length=160, return_tensors='pt').to('cuda:0')
    return torch.softmax(cmod(**enc).logits, -1)[:, 1].cpu().numpy()

print(f"\n{'Sub':<10} {'margin':>7} {'cur wR':>8} {'CE wR':>8} {'holdΔ':>8}  use")
ce_gate = {}
for sub in sorted(SUBSET_TO_LANG):
    tag, a, m = choice[sub]
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    tune, hold = idxs[0::2][:250], idxs[1::2][:250]
    def eval_ce(ix, margin):
        tot = []
        for i in ix:
            pool = val_cands_all[i]
            cq = [questions_raw[c['idx']] for c in pool]
            cs = ce_scores(val_qs[i], cq)
            b = int(np.argmax(cs))
            pick = pool[b]['answer'] if (b != 0 and cs[b]-cs[0] > margin) else pool[0]['answer']
            # CE replaces TOP-1 only; downstream comb_mbr/stitch unchanged for now
            rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]; at = uni_toks(pick)[:CAP]
            tot.append(0.37*uni_r1(rt, at)+0.37*uni_rl(rt, at))
        return np.mean(tot)
    def eval_base(ix):
        tot = []
        for i in ix:
            rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
            at = uni_toks(val_cands_all[i][0]['answer'])[:CAP]
            tot.append(0.37*uni_r1(rt, at)+0.37*uni_rl(rt, at))
        return np.mean(tot)
    best_m = max([0.1, 0.2, 0.3, 0.5], key=lambda mm: eval_ce(tune, mm))
    h_ce, h_b = eval_ce(hold, best_m), eval_base(hold)
    use = h_ce > h_b + 0.003
    ce_gate[sub] = (bool(use), best_m)
    print(f"{sub:<10} {best_m:>7.1f} {h_b:>8.4f} {h_ce:>8.4f} {h_ce-h_b:>+8.4f}  {'CE' if use else 'keep'}")
print("Gate:", ce_gate)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 152.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 39.7 MB/s eta 0:00:00
Pairs: 31625  (pos 11247)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
CE epoch 1:   0%|          | 0/1977 [00:00<?, ?it/s]/tmp/ipykernel_8741/32604022.py:60: UserWa

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Sub         margin   cur wR    CE wR    holdΔ  use
Aka_Gha        0.1   0.2003   0.2048  +0.0045  CE
Amh_Eth        0.2   0.1467   0.1539  +0.0072  CE
Eng_Eth        0.5   0.4961   0.4943  -0.0019  keep
Eng_Gha        0.3   0.1952   0.2017  +0.0065  CE
Eng_Ken        0.5   0.6463   0.6269  -0.0193  keep
Eng_Uga        0.5   0.6398   0.6267  -0.0131  keep
Lug_Uga        0.5   0.6080   0.5928  -0.0152  keep
Swa_Ken        0.5   0.6902   0.6693  -0.0209  keep
Gate: {'Aka_Gha': (True, 0.1), 'Amh_Eth': (True, 0.2), 'Eng_Eth': (False, 0.5), 'Eng_Gha': (True, 0.3), 'Eng_Ken': (False, 0.5), 'Eng_Uga': (False, 0.5), 'Lug_Uga': (False, 0.5), 'Swa_Ken': (False, 0.5)}


In [ ]:
# ==== CE-FED STRATEGIES: stitch + score-weighted MBR on CE-reranked pools ====
import numpy as np
from tqdm import tqdm

def ce_pool(i, sub):
    """Re-order the candidate pool by CE score; sim = CE prob (for MBR weighting)."""
    pool = val_cands_all[i]
    cq = [questions_raw[c['idx']] for c in pool]
    cs = ce_scores(val_qs[i], cq)
    order = np.argsort(-cs)
    return [{'answer': pool[j]['answer'], 'sim': float(cs[j]), 'idx': pool[j]['idx']}
            for j in order]

print(f"{'Sub':<10} {'deployed':>9} {'CE+stitch':>10} {'CE-wMBR':>8}  verdict")
for sub in ['Aka_Gha', 'Eng_Gha', 'Amh_Eth']:
    tag, a, m = choice[sub]
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    hold = idxs[1::2][:250]
    dep, st, wm = [], [], []
    g = uni_stitch_gate.get(sub)
    for i in tqdm(hold, desc=sub, leave=False):
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        cp = ce_pool(i, sub)
        def w(ans):
            at = uni_toks(ans)[:CAP]
            return 0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at)
        # deployed strategy baseline
        if sub == 'Eng_Gha':
            dep.append(0.2715)  # ft_gen holdout constant (per-query gens not indexed here)
        elif g and g['use']:
            dep.append(w(uni_stitch(val_cands_all[i], g['lam'], sub)))
        else:
            dd, ddw, u1, uL = uni_prep(val_cands_all[i])
            dep.append(w(dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]))
        # CE-fed stitch
        st.append(w(uni_stitch(cp, g['lam'] if g else 0.85, sub)))
        # CE-score-weighted MBR (sim = CE prob -> uni_prep's softmax weighting)
        dd, ddw, u1, uL = uni_prep(cp)
        wm.append(w(dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, 0.01)]))
    print(f"{sub:<10} {np.mean(dep):>9.4f} {np.mean(st):>10.4f} {np.mean(wm):>8.4f}  "
          f"{'CE-STITCH' if np.mean(st) > np.mean(dep)+0.004 else 'CE-wMBR' if np.mean(wm) > np.mean(dep)+0.004 else 'keep deployed'}")

Sub         deployed  CE+stitch  CE-wMBR  verdict


Aka_Gha       0.2208     0.2288   0.2081  CE-STITCH


Eng_Gha       0.2715     0.2128   0.1971  keep deployed


Amh_Eth       0.1491     0.1568   0.1427  CE-STITCH


In [ ]:
# =============================================================================
# V6: V4 + CE-stitch (Aka_Gha, Amh_Eth) + Lug_Uga adapter β=0.8 + Swa_Ken β=0.8
# Identical columns, open-source. Requires: bootstrap + cmod/ctok loaded.
# =============================================================================
import json, numpy as np, pandas as pd, torch
from tqdm import tqdm

ft2_corpus = np.load(CACHE/'ft2_corpus.npy'); ft2_test = np.load(CACHE/'ft2_test.npy')
pl_lug = np.load(CACHE/'pl_Lug_Uga_test.npy')
pl_lug_idx = json.load(open(CACHE/'pl_Lug_Uga_test_idx.json'))
pl_lug_map = {i: j for j, i in enumerate(pl_lug_idx)}
lug_corpus = np.load(CACHE/'pl_Lug_Uga_corpus.npy')   # saved during the adapter run
CE_LANGS = {'Aka_Gha', 'Amh_Eth'}

@torch.no_grad()
def ce_scores_t(query, cand_qs):
    enc = ctok([query]*len(cand_qs), cand_qs, padding=True, truncation=True,
               max_length=160, return_tensors='pt').to('cuda:0')
    return torch.softmax(cmod(**enc).logits, -1)[:, 1].cpu().numpy()

rows_v6 = []
v4 = pd.read_csv(OUTPUT_DIR/'submission_v4_final.csv')
v4_map = dict(zip(v4['ID'].astype(str), v4['TargetR1F1']))
for i in tqdm(range(len(test_df)), desc="V6"):
    rid = str(test_df.iloc[i]['ID']); sub = test_subs[i]
    if sub in CE_LANGS:
        pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                        k=K_CANDIDATES, exclude_exact=False)
        if pool:
            cs = ce_scores_t(test_qs[i], [questions_raw[c['idx']] for c in pool])
            order = np.argsort(-cs)
            cp = [{'answer': pool[j]['answer'], 'sim': float(cs[j]), 'idx': pool[j]['idx']}
                  for j in order]
            lam = uni_stitch_gate[sub]['lam'] if sub in uni_stitch_gate else 0.70
            ans = uni_stitch(cp, lam, sub)
        else:
            ans = v4_map[rid]
    elif sub == 'Lug_Uga':
        _, mask = lang_indices[sub]; mask_arr = np.array(mask)
        s = 0.8*(corpus_emb[mask_arr] @ test_emb[i]) + 0.2*(lug_corpus @ pl_lug[pl_lug_map[i]])
        order = np.argsort(-s)
        pool = [{'answer': answers_raw[int(mask_arr[j])], 'sim': float(s[j]),
                 'idx': int(mask_arr[j])} for j in order[:K_CANDIDATES]]
        tag, a, m = choice[sub]
        dd, ddw, u1, uL = uni_prep(pool)
        ans = dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]
    elif sub == 'Swa_Ken':
        _, mask = lang_indices[sub]; mask_arr = np.array(mask)
        s = 0.8*(corpus_emb[mask_arr] @ test_emb[i]) + 0.2*(ft2_corpus[mask_arr] @ ft2_test[i])
        order = np.argsort(-s)
        pool = [{'answer': answers_raw[int(mask_arr[j])], 'sim': float(s[j]),
                 'idx': int(mask_arr[j])} for j in order[:K_CANDIDATES]]
        tag, a, m = choice[sub]
        dd, ddw, u1, uL = uni_prep(pool)
        ans = dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]
    else:
        ans = v4_map[rid]    # everything else: byte-identical to V4
    rows_v6.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': ans,
                    'TargetRLF1': ans, 'TargetLLM': ans})

df6 = pd.DataFrame(rows_v6).reindex(columns=SUB_COLS)
assert len(df6) == len(test_df) and not df6.isnull().any().any()
df6.to_csv(OUTPUT_DIR/'submission_v6.csv', index=False)
print("Saved: submission_v6.csv")

V6: 100%|██████████| 2618/2618 [00:37<00:00, 70.71it/s] 


Saved: submission_v6.csv


In [ ]:
# =============================================================================
# CE-LARGE: xlm-roberta-large reranker, 2 epochs — upgrade path on the CE win
# Same mining as v2 (seeded), same guarded eval. ~2 hrs on L4.
# =============================================================================
import torch, gc, random, numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
random.seed(42); np.random.seed(42)

# ---- mining (identical to CE-v2; skip if `pairs` still in memory) ----
if 'pairs' not in dir():
    train_q_set = set(q.strip() for q in train_df['input'].dropna().astype(str))
    is_train_row = np.array([q in train_q_set for q in corpus_q_stripped])
    pairs = []
    for sub in SUBSET_TO_LANG:
        idx_t = [i for i in range(len(combined)) if subsets_raw[i]==sub and is_train_row[i]]
        random.shuffle(idx_t)
        ix, mask = lang_indices[sub]; mask_arr = np.array(mask)
        tmask = np.array([is_train_row[ci] for ci in mask])
        made = 0
        for qi in idx_t:
            if made >= 2500: break
            gold = uni_toks(answers_raw[qi])[:CAP]
            if len(gold) < 3: continue
            D, I = ix.search(corpus_emb[qi].reshape(1,-1), 20)
            pos, neg = [], []
            for li in I[0]:
                if li < 0 or not tmask[int(li)]: continue
                ci = int(mask_arr[int(li)])
                if ci == qi or corpus_q_stripped[ci] == corpus_q_stripped[qi]: continue
                r1 = uni_r1(gold, uni_toks(answers_raw[ci])[:CAP])
                (pos if r1 >= 0.5 else neg if r1 <= 0.2 else []).append(ci)
            if pos and neg:
                pairs.append((questions_raw[qi], questions_raw[random.choice(pos)], 1))
                for n_ in random.sample(neg, min(2, len(neg))):
                    pairs.append((questions_raw[qi], questions_raw[n_], 0))
                made += 1
    print(f"Pairs: {len(pairs)}")

gc.collect(); torch.cuda.empty_cache()
CE = 'xlm-roberta-large'
ctokL = AutoTokenizer.from_pretrained(CE)
cmodL = AutoModelForSequenceClassification.from_pretrained(CE, num_labels=2).to('cuda:0')
opt = torch.optim.AdamW(cmodL.parameters(), lr=1e-5)   # lower LR for large
B = 8
cmodL.train()
for ep in range(2):
    random.shuffle(pairs)
    pbar = tqdm(range(0, len(pairs), B), desc=f"CE-large ep{ep+1}")
    for s in pbar:
        chunk = pairs[s:s+B]
        enc = ctokL([p[0] for p in chunk], [p[1] for p in chunk], padding=True,
                    truncation=True, max_length=160, return_tensors='pt').to('cuda:0')
        labels = torch.tensor([p[2] for p in chunk]).to('cuda:0')
        out = cmodL(**enc, labels=labels)
        out.loss.backward(); opt.step(); opt.zero_grad()
        if s % (B*100) == 0: pbar.set_postfix(loss=float(out.loss.detach()))
cmodL.eval()
cmodL.save_pretrained(str(OUTPUT_DIR/'ce-reranker-large')); ctokL.save_pretrained(str(OUTPUT_DIR/'ce-reranker-large'))

# ---- guarded eval vs DEPLOYED strategies (not vs top-1 this time) ----
@torch.no_grad()
def ce_scores_L(query, cand_qs):
    enc = ctokL([query]*len(cand_qs), cand_qs, padding=True, truncation=True,
                max_length=160, return_tensors='pt').to('cuda:0')
    return torch.softmax(cmodL(**enc).logits, -1)[:, 1].cpu().numpy()

print(f"\n{'Sub':<10} {'deployed':>9} {'CEL-stitch':>10} {'CEL-top1':>9}  verdict")
for sub in sorted(SUBSET_TO_LANG):
    tag, a, m = choice[sub]
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    hold = idxs[1::2][:200]
    g = uni_stitch_gate.get(sub)
    dep, stl, t1l = [], [], []
    for i in tqdm(hold, desc=sub, leave=False):
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        def w(ans):
            at = uni_toks(ans)[:CAP]
            return 0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at)
        pool = val_cands_all[i]
        cs = ce_scores_L(val_qs[i], [questions_raw[c['idx']] for c in pool])
        order = np.argsort(-cs)
        cp = [{'answer': pool[j]['answer'], 'sim': float(cs[j]), 'idx': pool[j]['idx']} for j in order]
        # deployed baseline (V6 state)
        if sub == 'Eng_Gha': dep.append(0.2715)
        elif sub in ('Aka_Gha','Amh_Eth'):  # V6: CE-base-stitch ~ use stitch on base-CE pool? approx: deployed = stitch on plain pool
            dep.append(w(uni_stitch(pool, uni_stitch_gate.get(sub,{'lam':0.70})['lam'], sub)))
        elif g and g['use']: dep.append(w(uni_stitch(pool, g['lam'], sub)))
        else:
            dd, ddw, u1, uL = uni_prep(pool)
            dep.append(w(dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]))
        stl.append(w(uni_stitch(cp, uni_stitch_gate.get(sub,{'lam':0.85})['lam'], sub)))
        b = int(np.argmax(cs))
        pick = pool[b]['answer'] if (b != 0 and cs[b]-cs[0] > 0.3) else pool[0]['answer']
        t1l.append(w(pick))
    d, s_, t = np.mean(dep), np.mean(stl), np.mean(t1l)
    best = max(s_, t)
    print(f"{sub:<10} {d:>9.4f} {s_:>10.4f} {t:>9.4f}  "
          f"{'CEL-STITCH' if s_ > d + 0.004 else 'CEL-TOP1' if t > d + 0.004 else 'keep'}")

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

Eng_Gha       0.2715     0.2082    0.1940  keep


Eng_Ken       0.6463     0.2561    0.6463  keep


Eng_Uga       0.6384     0.4490    0.6384  keep


Lug_Uga       0.6188     0.3087    0.6188  keep


Swa_Ken       0.6849     0.2550    0.6849  keep


In [ ]:
# =============================================================================
# EXP 3: STRONG-LANG CE — duplication-group supervision (Eng_Uga, Eng_Ken, Swa_Ken, Lug_Uga)
# Positives: question pairs sharing an IDENTICAL answer (exact-match groups).
# Negatives: top-20 retrieval neighbors with different answers, answer-R1 <= 0.2.
# Train xlm-roberta-base (the proven config), guarded eval vs deployed top-1.
# =============================================================================
import torch, gc, random, numpy as np
from collections import defaultdict
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
random.seed(42); np.random.seed(42)

STRONG = ['Eng_Uga', 'Eng_Ken', 'Swa_Ken', 'Lug_Uga']
train_q_set = set(q.strip() for q in train_df['input'].dropna().astype(str))
is_train_row = np.array([q in train_q_set for q in corpus_q_stripped])

# ---- 1) answer-duplication groups (train rows, per language) ----
pairs2 = []
for sub in STRONG:
    groups = defaultdict(list)
    for i in range(len(combined)):
        if subsets_raw[i] == sub and is_train_row[i]:
            groups[answers_raw[i].strip()].append(i)
    dup_groups = [v for v in groups.values() if len(v) >= 2]
    ix, mask = lang_indices[sub]; mask_arr = np.array(mask)
    tmask = np.array([is_train_row[ci] for ci in mask])
    made = 0
    random.shuffle(dup_groups)
    for grp in dup_groups:
        if made >= 3000: break
        qi, pi = random.sample(grp, 2)                      # true positive pair
        gold = uni_toks(answers_raw[qi])[:CAP]
        D, I = ix.search(corpus_emb[qi].reshape(1,-1), 20)  # hard negatives
        negs = []
        for li in I[0]:
            if li < 0 or not tmask[int(li)]: continue
            ci = int(mask_arr[int(li)])
            if ci == qi or answers_raw[ci].strip() == answers_raw[qi].strip(): continue
            if uni_r1(gold, uni_toks(answers_raw[ci])[:CAP]) <= 0.2: negs.append(ci)
            if len(negs) >= 2: break
        if negs:
            pairs2.append((questions_raw[qi], questions_raw[pi], 1))
            for n_ in negs: pairs2.append((questions_raw[qi], questions_raw[n_], 0))
            made += 1
    print(f"{sub}: {made} groups -> dup-groups available: {len(dup_groups)}")
print(f"Total pairs: {len(pairs2)} (pos {sum(1 for p in pairs2 if p[2]==1)})")

# ---- 2) train (proven base config) ----
gc.collect(); torch.cuda.empty_cache()
ctok2 = AutoTokenizer.from_pretrained('xlm-roberta-base')
cmod2 = AutoModelForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=2).to('cuda:0')
opt = torch.optim.AdamW(cmod2.parameters(), lr=2e-5)
random.shuffle(pairs2); B = 16
cmod2.train()
for s in tqdm(range(0, len(pairs2), B), desc="CE-dup train"):
    chunk = pairs2[s:s+B]
    enc = ctok2([p[0] for p in chunk], [p[1] for p in chunk], padding=True,
                truncation=True, max_length=160, return_tensors='pt').to('cuda:0')
    labels = torch.tensor([p[2] for p in chunk]).to('cuda:0')
    out = cmod2(**enc, labels=labels)
    out.loss.backward(); opt.step(); opt.zero_grad()
cmod2.eval()
cmod2.save_pretrained(str(OUTPUT_DIR/'ce-dup-strong')); ctok2.save_pretrained(str(OUTPUT_DIR/'ce-dup-strong'))

# ---- 3) guarded eval vs deployed comb_mbr, margin-tuned, split-half ----
@torch.no_grad()
def ce2(query, cand_qs):
    enc = ctok2([query]*len(cand_qs), cand_qs, padding=True, truncation=True,
                max_length=160, return_tensors='pt').to('cuda:0')
    return torch.softmax(cmod2(**enc).logits, -1)[:, 1].cpu().numpy()

print(f"\n{'Sub':<10} {'margin':>7} {'deployed':>9} {'CE-dup':>8} {'holdΔ':>8}  use")
for sub in STRONG:
    tag, a, m = choice[sub]
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    tune, hold = idxs[0::2][:250], idxs[1::2][:250]
    def w_of(ans, rt):
        at = uni_toks(ans)[:CAP]
        return 0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at)
    def eval_dep(ix):
        out = []
        for i in ix:
            rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
            dd, ddw, u1, uL = uni_prep(val_cands_all[i])
            out.append(w_of(dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)], rt))
        return np.mean(out)
    def eval_ce(ix, margin):
        out = []
        for i in ix:
            rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
            pool = val_cands_all[i]
            cs = ce2(val_qs[i], [questions_raw[c['idx']] for c in pool])
            b = int(np.argmax(cs))
            pick = pool[b]['answer'] if (b != 0 and cs[b]-cs[0] > margin) else pool[0]['answer']
            out.append(w_of(pick, rt))
        return np.mean(out)
    bm = max([0.1, 0.2, 0.3, 0.5], key=lambda mm: eval_ce(tune, mm))
    h_ce, h_dep = eval_ce(hold, bm), eval_dep(hold)
    print(f"{sub:<10} {bm:>7.1f} {h_dep:>9.4f} {h_ce:>8.4f} {h_ce-h_dep:>+8.4f}  "
          f"{'CE-DUP' if h_ce > h_dep + 0.003 else 'keep'}")

Eng_Uga: 679 groups -> dup-groups available: 1028
Eng_Ken: 642 groups -> dup-groups available: 747
Swa_Ken: 606 groups -> dup-groups available: 747
Lug_Uga: 625 groups -> dup-groups available: 718
Total pairs: 7288 (pos 2552)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
CE-dup train: 100%|██████████| 456/456 [01:23<00:00,  5.46it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Sub         margin  deployed   CE-dup    holdΔ  use
Eng_Uga        0.5    0.6398   0.6386  -0.0012  keep
Eng_Ken        0.5    0.6463   0.6448  -0.0014  keep
Swa_Ken        0.5    0.6902   0.6716  -0.0186  keep
Lug_Uga        0.5    0.6080   0.6032  -0.0048  keep


In [ ]:
# =============================================================================
# CARD A: BGE-reranker-v2-m3 (pretrained, open-source) on weak-language pools
# Same guarded eval as CE — compares vs DEPLOYED (incl. V6's CE-stitch).
# =============================================================================
!pip install -q -U FlagEmbedding
# ==== BGE reranker via raw transformers (drop-in replacement) ====
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
btok = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')
bmod = AutoModelForSequenceClassification.from_pretrained(
    'BAAI/bge-reranker-v2-m3', dtype=torch.float16).to('cuda:0').eval()

@torch.no_grad()
def bge_scores(query, cand_questions):
    enc = btok([query]*len(cand_questions), cand_questions, padding=True,
               truncation=True, max_length=256, return_tensors='pt').to('cuda:0')
    return bmod(**enc).logits.squeeze(-1).float().cpu().numpy()

print(f"{'Sub':<10} {'deployed':>9} {'BGE-stitch':>10} {'BGE-top1':>9}  verdict")
for sub in ['Aka_Gha', 'Amh_Eth', 'Eng_Gha', 'Eng_Eth']:
    tag, a, m = choice[sub]
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    hold = idxs[1::2][:250]
    g = uni_stitch_gate.get(sub)
    dep, stl, t1l = [], [], []
    for i in tqdm(hold, desc=sub, leave=False):
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        def w(ans):
            at = uni_toks(ans)[:CAP]
            return 0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at)
        pool = val_cands_all[i]
        cs = bge_scores(val_qs[i], [questions_raw[c['idx']] for c in pool])
        order = np.argsort(-cs)
        # sims for stitch/MBR weighting: softmax of BGE logits
        e = np.exp(cs - cs.max()); p = e / e.sum()
        cp = [{'answer': pool[j]['answer'], 'sim': float(p[j]), 'idx': pool[j]['idx']} for j in order]
        # deployed baselines: V6 state
        if sub == 'Eng_Gha': dep.append(0.2715)
        elif sub in ('Aka_Gha', 'Amh_Eth'):
            cq = [questions_raw[c['idx']] for c in pool]
            cs2 = ce_scores(val_qs[i], cq)                       # CE-base from this morning
            o2 = np.argsort(-cs2)
            cp2 = [{'answer': pool[j]['answer'], 'sim': float(cs2[j]), 'idx': pool[j]['idx']} for j in o2]
            dep.append(w(uni_stitch(cp2, uni_stitch_gate.get(sub, {'lam':0.70})['lam'], sub)))
        elif g and g['use']: dep.append(w(uni_stitch(pool, g['lam'], sub)))
        else:
            dd, ddw, u1, uL = uni_prep(pool)
            dep.append(w(dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]))
        stl.append(w(uni_stitch(cp, uni_stitch_gate.get(sub, {'lam':0.85})['lam'], sub)))
        b = int(np.argmax(cs))
        t1l.append(w(pool[b]['answer'] if b != 0 and (cs[b]-cs[0]) > 1.0 else pool[0]['answer']))
    d, s_, t = np.mean(dep), np.mean(stl), np.mean(t1l)
    print(f"{sub:<10} {d:>9.4f} {s_:>10.4f} {t:>9.4f}  "
          f"{'BGE-STITCH' if s_ > d + 0.004 else 'BGE-TOP1' if t > d + 0.004 else 'keep'}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Sub         deployed BGE-stitch  BGE-top1  verdict


Aka_Gha       0.2288     0.2215    0.2057  keep


Amh_Eth       0.1568     0.1497    0.1581  keep


Eng_Gha       0.2715     0.2071    0.2045  keep


Eng_Eth       0.4961     0.4027    0.4650  keep


In [ ]:
# ==== CROSS-LINGUAL POOL: Eng_Gha queries -> Aka_Gha answer space, oracle test ====
import numpy as np
from tqdm import tqdm
_, aka_mask = lang_indices['Aka_Gha']; aka_arr = np.array(aka_mask)
sub = 'Eng_Gha'
idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
        and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
hold = idxs[1::2][:300]
o_same, o_union = [], []
for i in tqdm(hold, desc="xling oracle"):
    rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
    def w(ans):
        at = uni_toks(ans)[:CAP]
        return 0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at)
    o_same.append(max(w(c['answer']) for c in val_cands_all[i]))
    s = corpus_emb[aka_arr] @ val_emb[i]
    aka_pool = [answers_raw[int(aka_arr[j])] for j in np.argsort(-s)[:15]]
    o_union.append(max(o_same[-1], max(w(a_) for a_ in aka_pool)))
print(f"\nEng_Gha pool oracle: same-lang={np.mean(o_same):.4f}  +Akan-answers={np.mean(o_union):.4f}  "
      f"Δ={np.mean(o_union)-np.mean(o_same):+.4f}")
print("Δ > +0.03 -> Akan answers contain real Eng_Gha coverage -> worth a deploy experiment.")
print("Δ small  -> the 'topics exist only in Akan' claim doesn't convert to ROUGE -> closed.")

xling oracle: 100%|██████████| 300/300 [00:04<00:00, 63.04it/s]


Eng_Gha pool oracle: same-lang=0.2566  +Akan-answers=0.2568  Δ=+0.0001
Δ > +0.03 -> Akan answers contain real Eng_Gha coverage -> worth a deploy experiment.
Δ small  -> the 'topics exist only in Akan' claim doesn't convert to ROUGE -> closed.


In [ ]:
# =============================================================================
# CARD B (final stone): Q -> (Q+A concatenated) retrieval leg, weak languages
# Encode corpus as "question [SEP-ish] answer", search with query embeddings,
# union with existing pool, test oracle + CE-stitch vs deployed.
# =============================================================================
import torch, gc, numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
gc.collect(); torch.cuda.empty_cache()
enc_model = SentenceTransformer(str(OUTPUT_DIR/'afrie5-final-model'), device='cuda:0')
enc_model.max_seq_length = 256

WEAK = ['Aka_Gha', 'Amh_Eth', 'Eng_Gha']
print(f"{'Sub':<10} {'oracle cur':>10} {'oracle +QA':>10} {'dep':>8} {'QA-CE-stitch':>12}  verdict")
for sub in WEAK:
    _, mask = lang_indices[sub]; mask_arr = np.array(mask)
    qa_texts = [f"passage: {questions_raw[ci]} {answers_raw[ci]}" for ci in mask_arr]
    qa_emb = enc_model.encode(qa_texts, batch_size=32, normalize_embeddings=True,
                              show_progress_bar=True).astype(np.float32)
    tag, a, m = choice[sub]
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    hold = idxs[1::2][:250]
    g_lam = uni_stitch_gate.get(sub, {'lam': 0.70})['lam']
    o_cur, o_qa, dep, qa_ce = [], [], [], []
    for i in tqdm(hold, desc=sub, leave=False):
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        def w(ans):
            at = uni_toks(ans)[:CAP]
            return 0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at)
        pool = val_cands_all[i]
        o_cur.append(max(w(c['answer']) for c in pool))
        s = qa_emb @ val_emb[i]
        q = val_qs[i].strip()
        qa_pool = []
        for j in np.argsort(-s)[:15]:
            ci = int(mask_arr[j])
            if corpus_q_stripped[ci] == q: continue
            qa_pool.append({'answer': answers_raw[ci], 'sim': float(s[j]), 'idx': ci})
        # union, dedup by idx
        seen = {c['idx'] for c in pool}
        upool = pool + [c for c in qa_pool if c['idx'] not in seen]
        o_qa.append(max(w(c['answer']) for c in upool))
        # deployed baseline (V6)
        if sub == 'Eng_Gha': dep.append(0.2715)
        else:
            cs0 = ce_scores(val_qs[i], [questions_raw[c['idx']] for c in pool])
            cp0 = [{'answer': pool[j]['answer'], 'sim': float(cs0[j]), 'idx': pool[j]['idx']}
                   for j in np.argsort(-cs0)]
            dep.append(w(uni_stitch(cp0, g_lam, sub)))
        # challenger: CE-stitch on the UNION pool
        cs1 = ce_scores(val_qs[i], [questions_raw[c['idx']] for c in upool])
        cp1 = [{'answer': upool[j]['answer'], 'sim': float(cs1[j]), 'idx': upool[j]['idx']}
               for j in np.argsort(-cs1)]
        qa_ce.append(w(uni_stitch(cp1, g_lam, sub)))
    oc, oq, d, qc = map(np.mean, (o_cur, o_qa, dep, qa_ce))
    print(f"{sub:<10} {oc:>10.4f} {oq:>10.4f} {d:>8.4f} {qc:>12.4f}  "
          f"{'QA-UNION' if qc > d + 0.004 else 'keep'}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Sub        oracle cur oracle +QA      dep QA-CE-stitch  verdict


Batches:   0%|          | 0/175 [00:00<?, ?it/s]

Aka_Gha        0.2625     0.2663   0.2288       0.2296  keep


Batches:   0%|          | 0/73 [00:00<?, ?it/s]

Amh_Eth        0.2396     0.2472   0.1568       0.1610  QA-UNION


Batches:   0%|          | 0/174 [00:00<?, ?it/s]

Eng_Gha        0.2571     0.2633   0.2715       0.2154  keep


In [ ]:
# =============================================================================
# V7 = V6 + Amh_Eth QA-union CE-stitch (only ~60 rows change)
# Requires: enc_model (AfriE5), cmod/ctok (ce-reranker-v2), bootstrap state.
# =============================================================================
import numpy as np, pandas as pd, torch
from tqdm import tqdm

sub = 'Amh_Eth'
_, mask = lang_indices[sub]; mask_arr = np.array(mask)
qa_texts = [f"passage: {questions_raw[ci]} {answers_raw[ci]}" for ci in mask_arr]
qa_emb = enc_model.encode(qa_texts, batch_size=32, normalize_embeddings=True,
                          show_progress_bar=True).astype(np.float32)
np.save(CACHE/'qa_emb_amh.npy', qa_emb)   # for reproducibility package

g_lam = uni_stitch_gate.get(sub, {'lam': 0.70})['lam']
t_idx = [i for i in range(len(test_df)) if test_subs[i] == sub]
new_ans = {}
for i in tqdm(t_idx, desc="Amh V7"):
    pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                    k=K_CANDIDATES, exclude_exact=False)
    s = qa_emb @ test_emb[i]
    qa_pool = [{'answer': answers_raw[int(mask_arr[j])], 'sim': float(s[j]),
                'idx': int(mask_arr[j])} for j in np.argsort(-s)[:15]]
    seen = {c['idx'] for c in pool}
    upool = pool + [c for c in qa_pool if c['idx'] not in seen]
    if not upool: continue
    cs = ce_scores(test_qs[i], [questions_raw[c['idx']] for c in upool])
    cp = [{'answer': upool[j]['answer'], 'sim': float(cs[j]), 'idx': upool[j]['idx']}
          for j in np.argsort(-cs)]
    new_ans[str(test_df.iloc[i]['ID'])] = uni_stitch(cp, g_lam, sub)

v6 = pd.read_csv(OUTPUT_DIR/'submission_v6.csv')
sel = v6['ID'].astype(str).isin(new_ans)
for col in ['TargetR1F1','TargetRLF1','TargetLLM']:
    v6.loc[sel, col] = v6.loc[sel, 'ID'].astype(str).map(new_ans)
assert len(v6) == len(test_df) and not v6.isnull().any().any()
print(f"Rows changed: {sel.sum()}")
v6.to_csv(OUTPUT_DIR/'submission_v7.csv', index=False)
print("Saved: submission_v7.csv")

Batches:   0%|          | 0/73 [00:00<?, ?it/s]

Amh V7: 100%|██████████| 61/61 [00:03<00:00, 18.57it/s]


Rows changed: 61
Saved: submission_v7.csv


In [ ]:
# ==== EXP 1 FIXED: leak-proof pools (exact-question filter restored) ====
import numpy as np
from tqdm import tqdm
LAMS = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

for sub in ['Aka_Gha', 'Amh_Eth']:
    _, mask = lang_indices[sub]; mask_arr = np.array(mask)
    qa_texts = [f"passage: {questions_raw[ci]} {answers_raw[ci]}" for ci in mask_arr]
    qa_emb = enc_model.encode(qa_texts, batch_size=32, normalize_embeddings=True,
                              show_progress_bar=False).astype(np.float32)
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    tune, hold = idxs[0::2][:250], idxs[1::2][:250]
    dep_lam = uni_stitch_gate.get(sub, {'lam': 0.70})['lam']
    def build_pool(i):
        q = val_qs[i].strip()
        pool = val_cands_all[i]
        s = qa_emb @ val_emb[i]
        seen = {c['idx'] for c in pool}
        qa_pool = []
        for j in np.argsort(-s)[:20]:
            ci = int(mask_arr[j])
            if corpus_q_stripped[ci] == q: continue        # <-- the missing line
            if ci in seen: continue
            qa_pool.append({'answer': answers_raw[ci], 'sim': float(s[j]), 'idx': ci})
            if len(qa_pool) >= 15: break
        pool = pool + qa_pool
        cs = ce_scores(val_qs[i], [questions_raw[c['idx']] for c in pool])
        return [{'answer': pool[j]['answer'], 'sim': float(cs[j]), 'idx': pool[j]['idx']}
                for j in np.argsort(-cs)]
    pools_cache = {i: build_pool(i) for i in tqdm(tune+hold, desc=f"{sub} pools")}
    def wsc(ix, lam):
        out = []
        for i in ix:
            rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
            at = uni_toks(uni_stitch(pools_cache[i], lam, sub))[:CAP]
            out.append(0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at))
        return np.mean(out)
    best_lam = max(LAMS, key=lambda L: wsc(tune, L))
    h_new, h_dep = wsc(hold, best_lam), wsc(hold, dep_lam)
    print(f"{sub}: dep λ={dep_lam} -> {h_dep:.4f} | re-tuned λ={best_lam} -> {h_new:.4f} | "
          f"Δ={h_new-h_dep:+.4f} {'ADOPT' if h_new > h_dep + 0.003 else 'keep'}")

Aka_Gha pools: 100%|██████████| 500/500 [00:58<00:00,  8.51it/s]


Aka_Gha: dep λ=0.7 -> 0.2275 | re-tuned λ=0.55 -> 0.2279 | Δ=+0.0004 keep


Amh_Eth pools: 100%|██████████| 462/462 [00:22<00:00, 20.12it/s]


Amh_Eth: dep λ=0.7 -> 0.1617 | re-tuned λ=0.75 -> 0.1605 | Δ=-0.0012 keep


In [ ]:
import gc, torch
for nm in ['bmod', 'btok', 'cmodL', 'ctokL', 'cmod2', 'ctok2', 'rr']:
    if nm in dir(): exec(f"del {nm}")
gc.collect(); torch.cuda.empty_cache()
print(f"free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")   # want ~17+ GB

free: 14.6 GB


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
FT_MODEL_DIR = OUTPUT_DIR / 'qwen-ft-health'
tok = AutoTokenizer.from_pretrained(str(FT_MODEL_DIR))
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-7B-Instruct',
        dtype=torch.bfloat16, device_map='cuda:0')
ft = PeftModel.from_pretrained(base, str(FT_MODEL_DIR)); ft.eval()

@torch.no_grad()
def ft_generate(q, lang, cands, max_new=350):
    ctx = "\n".join(f"{k+1}. {c['answer']}" for k, c in enumerate(cands[:5]))
    msgs = [{"role":"system","content":
             f"You are a multilingual health expert. Answer health questions based on the "
             f"reference information provided. Use the EXACT words and phrases from the "
             f"references when possible. Be complete and accurate. Answer in {lang}."},
            {"role":"user","content": f"Question: {q}\n\nReference answers:\n{ctx}"}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to('cuda:0')
    out = ft.generate(**enc, max_new_tokens=max_new, do_sample=False,
                      pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print(f"loaded — free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

loaded — free: 0.7 GB


In [ ]:
# =============================================================================
# EXP 2: Eng_Gha ft_gen with QA-union references (val holdout, ~280 gens)
# Requires: ft/tok (bf16 qwen-ft-health), enc_model, cmod/ctok, bootstrap.
# =============================================================================
import json, numpy as np, torch
from tqdm import tqdm
sub = 'Eng_Gha'
_, mask = lang_indices[sub]; mask_arr = np.array(mask)
qa_texts = [f"passage: {questions_raw[ci]} {answers_raw[ci]}" for ci in mask_arr]
qa_emb = enc_model.encode(qa_texts, batch_size=32, normalize_embeddings=True,
                          show_progress_bar=True).astype(np.float32)

idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
        and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
hold = idxs[1::2][:280]

def union_pool(i):
    q = val_qs[i].strip()
    pool = list(val_cands_all[i])
    seen = {c['idx'] for c in pool}
    s = qa_emb @ val_emb[i]
    for j in np.argsort(-s)[:20]:
        ci = int(mask_arr[j])
        if corpus_q_stripped[ci] == q or ci in seen: continue
        pool.append({'answer': answers_raw[ci], 'sim': float(s[j]), 'idx': ci})
        if len(pool) >= 25: break
    cs = ce_scores(val_qs[i], [questions_raw[c['idx']] for c in pool])
    return [pool[j] for j in np.argsort(-cs)]      # CE-ordered union: top-5 feed the prompt

gp = OUTPUT_DIR / 'ftqwen_enggha_qaunion_val.json'
ga = json.load(open(gp)) if gp.exists() else {}
for n, i in enumerate(tqdm(hold, desc="EngGha QA-union gen")):
    if str(i) in ga: continue
    ga[str(i)] = ft_generate(val_qs[i], 'English (Ghana)', union_pool(i))
    if (n+1) % 25 == 0: json.dump(ga, open(gp, 'w'))
json.dump(ga, open(gp, 'w'))

scores = []
for i in hold:
    rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
    at = uni_toks(ga[str(i)])[:CAP]
    scores.append(0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at))
print(f"\nEng_Gha QA-union gen: {np.mean(scores):.4f}  vs deployed 0.2715  "
      f"Δ={np.mean(scores)-0.2715:+.4f}  {'ADOPT' if np.mean(scores) > 0.2715 + 0.004 else 'keep'}")

Batches:   0%|          | 0/174 [00:00<?, ?it/s]

EngGha QA-union gen: 100%|██████████| 280/280 [35:08<00:00,  7.53s/it]


Eng_Gha QA-union gen: 0.2886  vs deployed 0.2715  Δ=+0.0171  ADOPT


In [4]:
# ==== V8 PREP: reload AfriE5 + CE + Qwen (in this order) ====
import torch, gc
from sentence_transformers import SentenceTransformer
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForCausalLM)
from peft import PeftModel
gc.collect(); torch.cuda.empty_cache()

# 1) AfriE5 encoder (for qa_emb)
enc_model = SentenceTransformer(str(OUTPUT_DIR/'afrie5-final-model'), device='cuda:0')
enc_model.max_seq_length = 256

# 2) CE reranker (for ce_scores)
ctok = AutoTokenizer.from_pretrained(str(OUTPUT_DIR/'ce-reranker-v2'))
cmod = AutoModelForSequenceClassification.from_pretrained(
    str(OUTPUT_DIR/'ce-reranker-v2')).to('cuda:0').eval()
@torch.no_grad()
def ce_scores(query, cand_qs):
    enc = ctok([query]*len(cand_qs), cand_qs, padding=True, truncation=True,
               max_length=160, return_tensors='pt').to('cuda:0')
    return torch.softmax(cmod(**enc).logits, -1)[:, 1].cpu().numpy()

# 3) FT Qwen (for ft_generate)
FT_MODEL_DIR = OUTPUT_DIR / 'qwen-ft-health'
tok = AutoTokenizer.from_pretrained(str(FT_MODEL_DIR))
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-7B-Instruct',
        dtype=torch.bfloat16, device_map='cuda:0')
ft = PeftModel.from_pretrained(base, str(FT_MODEL_DIR)); ft.eval()

@torch.no_grad()
def ft_generate(q, lang, cands, max_new=350):
    ctx = "\n".join(f"{k+1}. {c['answer']}" for k, c in enumerate(cands[:5]))
    msgs = [{"role":"system","content":
             f"You are a multilingual health expert. Answer health questions based on the "
             f"reference information provided. Use the EXACT words and phrases from the "
             f"references when possible. Be complete and accurate. Answer in {lang}."},
            {"role":"user","content": f"Question: {q}\n\nReference answers:\n{ctx}"}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to('cuda:0')
    out = ft.generate(**enc, max_new_tokens=max_new, do_sample=False,
                      pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(f"all loaded — free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

all loaded — free: 4.5 GB


In [5]:
# =============================================================================
# V8: Eng_Gha test regeneration with QA-union references, then patch V7 -> V8
# =============================================================================
import json, numpy as np, pandas as pd, torch
from tqdm import tqdm

sub = 'Eng_Gha'
_, mask = lang_indices[sub]; mask_arr = np.array(mask)
qa_texts = [f"passage: {questions_raw[ci]} {answers_raw[ci]}" for ci in mask_arr]
qa_emb_t = enc_model.encode(qa_texts, batch_size=32, normalize_embeddings=True,
                            show_progress_bar=True).astype(np.float32)
np.save(CACHE/'qa_emb_enggha.npy', qa_emb_t)        # reproducibility

def union_pool_test(i):
    pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                    k=K_CANDIDATES, exclude_exact=False)
    seen = {c['idx'] for c in pool}
    s = qa_emb_t @ test_emb[i]
    for j in np.argsort(-s)[:20]:
        ci = int(mask_arr[j])
        if ci in seen: continue
        pool.append({'answer': answers_raw[ci], 'sim': float(s[j]), 'idx': ci})
        if len(pool) >= 25: break
    if not pool: return []
    cs = ce_scores(test_qs[i], [questions_raw[c['idx']] for c in pool])
    return [pool[j] for j in np.argsort(-cs)]

gp = OUTPUT_DIR / 'ftqwen_enggha_qaunion_test.json'
ga = json.load(open(gp)) if gp.exists() else {}
t_idx = [i for i in range(len(test_df)) if test_subs[i] == sub]
print(f"Eng_Gha test queries: {len(t_idx)}")
for n, i in enumerate(tqdm(t_idx, desc="V8 EngGha test gen")):
    rid = str(test_df.iloc[i]['ID'])
    if rid in ga: continue
    up = union_pool_test(i)
    ga[rid] = ft_generate(test_qs[i], 'English (Ghana)', up) if up else "No answer."
    if (n+1) % 25 == 0: json.dump(ga, open(gp, 'w'))
json.dump(ga, open(gp, 'w'))

v7 = pd.read_csv(OUTPUT_DIR/'submission_v7.csv')
sel = v7['ID'].astype(str).isin(ga)
for col in ['TargetR1F1','TargetRLF1','TargetLLM']:
    v7.loc[sel, col] = v7.loc[sel, 'ID'].astype(str).map(ga)
assert len(v7) == len(test_df) and not v7.isnull().any().any()
print(f"Rows changed: {sel.sum()}")
v7.to_csv(OUTPUT_DIR/'submission_v8.csv', index=False)
print("Saved: submission_v8.csv")

Batches:   0%|          | 0/174 [00:00<?, ?it/s]

Eng_Gha test queries: 491


V8 EngGha test gen: 100%|██████████| 491/491 [1:01:40<00:00,  7.54s/it]


Rows changed: 491
Saved: submission_v8.csv


In [6]:
# =============================================================================
# BEAM-STITCH vs GREEDY: identical objective, better search. Split-half eval.
# Requires: bootstrap + ce_scores + enc_model (all loaded in V8 session).
# =============================================================================
import numpy as np
from collections import Counter
from tqdm import tqdm

def beam_stitch(cands, lam, sub, beam=8, max_depth=14):
    answers = [c['answer'] for c in cands]
    w = np.exp(np.array([c['sim'] for c in cands])*5); w /= w.sum()
    p = Counter()
    for a, wi in zip(answers, w):
        for t, c in Counter(uni_toks(a)[:CAP]).items(): p[t] += wi*c
    ref_len = max(lam*uni_prior[sub], 1.0)
    seen, pool = set(), []
    for a in answers:
        for s in _re.split(r'(?<=[.!?])\s+|\n+', a):
            s = s.strip(); st = uni_toks(s)
            if len(st) < 4: continue
            k = ' '.join(st)
            if k not in seen: seen.add(k); pool.append((s, Counter(st), len(st)))
    if not pool: return answers[0]
    def ef1(cov, hl):
        mt = sum(min(c, p[t]) for t, c in cov.items())
        if hl == 0 or mt == 0: return 0.0
        pr_, rc = mt/hl, mt/ref_len; return 2*pr_*rc/(pr_+rc)
    best_sel, best_f1 = (), 0.0
    states = [((), Counter(), 0)]
    for _ in range(max_depth):
        nxt, seen_sets = [], set()
        for sel, cov, hl in states:
            used = set(sel)
            for i, (s, sc_, sl) in enumerate(pool):
                if i in used: continue
                key = tuple(sorted(used | {i}))
                if key in seen_sets: continue
                seen_sets.add(key)
                nc = cov.copy(); nc.update(sc_)
                nxt.append((key, nc, hl+sl, ef1(nc, hl+sl)))
        if not nxt: break
        nxt.sort(key=lambda x: -x[3])
        if nxt[0][3] <= best_f1 and len(states[0][0]) >= 2: break   # plateau stop
        states = [(k, c, h) for k, c, h, _ in nxt[:beam]]
        if nxt[0][3] > best_f1: best_f1, best_sel = nxt[0][3], nxt[0][0]
    if not best_sel: return answers[0]
    return ' '.join(pool[i][0] for i in sorted(best_sel))   # emit in pool order

# ---- eval: greedy vs beam on identical deployed-type pools ----
sub_qaemb = {}
for sub in ['Amh_Eth']:
    _, mask = lang_indices[sub]; mask_arr = np.array(mask)
    qa_texts = [f"passage: {questions_raw[ci]} {answers_raw[ci]}" for ci in mask_arr]
    sub_qaemb[sub] = (enc_model.encode(qa_texts, batch_size=32, normalize_embeddings=True,
                      show_progress_bar=False).astype(np.float32), mask_arr)

def deployed_pool(i, sub):
    q = val_qs[i].strip()
    pool = list(val_cands_all[i])
    if sub in sub_qaemb:                                   # Amh: QA-union first
        qa_emb_, mask_arr = sub_qaemb[sub]
        seen = {c['idx'] for c in pool}
        s = qa_emb_ @ val_emb[i]
        for j in np.argsort(-s)[:20]:
            ci = int(mask_arr[j])
            if corpus_q_stripped[ci] == q or ci in seen: continue
            pool.append({'answer': answers_raw[ci], 'sim': float(s[j]), 'idx': ci})
            if len(pool) >= 25: break
    cs = ce_scores(val_qs[i], [questions_raw[c['idx']] for c in pool])
    return [{'answer': pool[j]['answer'], 'sim': float(cs[j]), 'idx': pool[j]['idx']}
            for j in np.argsort(-cs)]

print(f"{'Sub':<10} {'greedy':>8} {'beam':>8} {'delta':>8}  verdict")
for sub in ['Aka_Gha', 'Amh_Eth']:
    lam = uni_stitch_gate[sub]['lam']
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    hold = idxs[1::2][:250]
    gs, bs = [], []
    for i in tqdm(hold, desc=sub, leave=False):
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        cp = deployed_pool(i, sub)
        def w(ans):
            at = uni_toks(ans)[:CAP]
            return 0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at)
        gs.append(w(uni_stitch(cp, lam, sub)))
        bs.append(w(beam_stitch(cp, lam, sub)))
    d = np.mean(bs) - np.mean(gs)
    print(f"{sub:<10} {np.mean(gs):>8.4f} {np.mean(bs):>8.4f} {d:>+8.4f}  "
          f"{'BEAM' if d > 0.003 else 'keep greedy'}")

Sub          greedy     beam    delta  verdict


Aka_Gha      0.2288   0.2305  +0.0017  keep greedy


Amh_Eth      0.1622   0.1610  -0.0011  keep greedy


In [7]:
# =============================================================================
# QA-UNION GENERATION PROBES: Amh_Eth + Aka_Gha (val holdout, resumable)
# Bars: Amh_Eth 0.1622 (QA-union CE-stitch, deployed in V8)
#       Aka_Gha 0.2288 (CE-stitch, deployed in V8)
# =============================================================================
import json, numpy as np, torch
from tqdm import tqdm

PROBES = {'Amh_Eth': ('Amharic (Ethiopia)', 0.1622, 200),
          'Aka_Gha': ('Akan (Ghana)',       0.2288, 250)}

for sub, (lang, BAR, N) in PROBES.items():
    _, mask = lang_indices[sub]; mask_arr = np.array(mask)
    qa_texts = [f"passage: {questions_raw[ci]} {answers_raw[ci]}" for ci in mask_arr]
    qa_emb_ = enc_model.encode(qa_texts, batch_size=32, normalize_embeddings=True,
                               show_progress_bar=False).astype(np.float32)
    def union_pool(i):
        q = val_qs[i].strip()
        pool = list(val_cands_all[i])
        seen = {c['idx'] for c in pool}
        s = qa_emb_ @ val_emb[i]
        for j in np.argsort(-s)[:20]:
            ci = int(mask_arr[j])
            if corpus_q_stripped[ci] == q or ci in seen: continue
            pool.append({'answer': answers_raw[ci], 'sim': float(s[j]), 'idx': ci})
            if len(pool) >= 25: break
        cs = ce_scores(val_qs[i], [questions_raw[c['idx']] for c in pool])
        return [pool[j] for j in np.argsort(-cs)]
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    hold = idxs[1::2][:N]
    gp = OUTPUT_DIR / f'ftqwen_{sub.lower()}_qaunion_val.json'
    ga = json.load(open(gp)) if gp.exists() else {}
    for n, i in enumerate(tqdm(hold, desc=f"{sub} QA-union gen")):
        if str(i) in ga: continue
        ga[str(i)] = ft_generate(val_qs[i], lang, union_pool(i))
        if (n+1) % 25 == 0: json.dump(ga, open(gp, 'w'))
    json.dump(ga, open(gp, 'w'))
    scores = []
    for i in hold:
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        at = uni_toks(ga[str(i)])[:CAP]
        scores.append(0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at))
    s = np.mean(scores)
    print(f"\n{sub}: QA-union gen = {s:.4f}  vs deployed {BAR}  Δ={s-BAR:+.4f}  "
          f"{'ADOPT' if s > BAR + 0.004 else 'keep'}\n")

Amh_Eth QA-union gen: 100%|██████████| 200/200 [39:31<00:00, 11.86s/it]



Amh_Eth: QA-union gen = 0.1336  vs deployed 0.1622  Δ=-0.0286  keep



Aka_Gha QA-union gen: 100%|██████████| 250/250 [1:31:07<00:00, 21.87s/it]


Aka_Gha: QA-union gen = 0.2140  vs deployed 0.2288  Δ=-0.0148  keep

